# 07 Advanced Challenge - IC50-style Curve Fitting in R

## Biochemistry question

In this synthetic dose-response dataset, what educational IC50-style pattern is suggested by a simple fitted curve?

This uses base R `nls()` for a simple 4-parameter logistic fit. It is a learning example, not a production pharmacology workflow.


In [ ]:
library(tidyverse)

df <- read_csv("../data/dose_response_ic50_sample.csv")

summary <- df %>%
  group_by(drug_name, concentration_uM) %>%
  summarise(
    mean_viability = mean(cell_viability_percent),
    sd_viability = sd(cell_viability_percent),
    n = n(),
    sem_viability = sd_viability / sqrt(n),
    .groups = "drop"
  )

summary

In [ ]:
fit_one_drug <- function(drug_label) {
  d <- summary %>%
    filter(drug_name == drug_label, concentration_uM > 0)

  model <- nls(
    mean_viability ~ bottom + (top - bottom) / (1 + (concentration_uM / ic50)^hill),
    data = d,
    start = list(
      bottom = min(d$mean_viability),
      top = max(d$mean_viability),
      ic50 = median(d$concentration_uM),
      hill = 1
    ),
    control = nls.control(maxiter = 200)
  )

  coef(model)
}

fit_x <- fit_one_drug("CompoundX")
fit_y <- fit_one_drug("CompoundY")

fit_x
fit_y

In [ ]:
# Prepare predicted curves.
make_curve <- function(drug_label, coefs) {
  x_vals <- exp(seq(log(0.003), log(3), length.out = 200))
  tibble(
    drug_name = drug_label,
    concentration_uM = x_vals,
    predicted = coefs["bottom"] + (coefs["top"] - coefs["bottom"]) / (1 + (x_vals / coefs["ic50"])^coefs["hill"])
  )
}

curve_df <- bind_rows(
  make_curve("CompoundX", fit_x),
  make_curve("CompoundY", fit_y)
)

ggplot() +
  geom_point(data = summary %>% filter(concentration_uM > 0),
             aes(x = concentration_uM, y = mean_viability, color = drug_name)) +
  geom_line(data = curve_df,
            aes(x = concentration_uM, y = predicted, color = drug_name)) +
  scale_x_log10() +
  labs(
    title = "IC50-style Curve Fitting Challenge",
    x = "Concentration (uM, log scale)",
    y = "Mean Cell Viability (%)"
  )

## Interpretation Questions

1. Which compound has the lower educational IC50-style estimate?
2. Which curve fitting code feels easier to understand compared with Python?
3. Why should this estimate be treated cautiously?
4. What would improve the reliability of the fit?

## Limitations

- This is synthetic data for learning curve-fitting ideas.
- The fitted value is not drug potency, clinical, diagnostic, regulatory, or efficacy evidence.
- Real IC50 analysis requires stronger experimental design, replicate structure, model checking, and domain review.
